# 05 - Results, Figures, and Honest Reporting

Load the latest `results/metrics/summary.json` and display only metrics that were actually recorded. If the summary is absent, the notebook says so instead of inventing values.

The final quick suite is a **single-seed synthetic** run; no significance claim is made.

In [ ]:
from pathlib import Path
import json
import sys

# Headless-safe plotting; Jupyter still renders figures inline.
import matplotlib
matplotlib.use("Agg")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


In [ ]:
import pandas as pd
from IPython.display import Image, display, Markdown

summary_path = ROOT / "results" / "metrics" / "summary.json"
if not summary_path.exists():
    print("No experimental result available. Run: python main.py --demo or python main.py --mode quick")
    summary = None
else:
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("Suite:", summary.get("suite_id"))
    print("Mode:", summary.get("mode"), "| Demo:", summary.get("demo"), "| Seeds:", summary.get("seeds"))
    print("Data:", summary.get("dataset", {}).get("source"))
    print("\nStages:")
    display(pd.DataFrame([{"stage": k, "status": v.get("status")} for k, v in summary.get("stages", {}).items()]))  # noqa: F821


## Model and federated comparison

In [ ]:
if summary:
    for group in ["centralized", "fedavg", "proposed"]:
        rows = summary.get("main", {}).get(group, [])
        if rows:
            print("\n" + group.upper())
            frame = pd.DataFrame(rows)
            cols = [c for c in ["method", "precision", "recall", "f1", "roc_auc", "n_params", "train_time_s", "total_bytes"] if c in frame.columns]
            display(frame[cols])  # noqa: F821


## Measured communication reduction

In [ ]:
if summary:
    cc = summary.get("main", {}).get("communication_comparison", {})
    print(json.dumps(cc, indent=2))
    if cc.get("reduction_pct") is not None:
        print(f"Measured reduction: {cc['reduction_pct']:.2f}%")


## Sweeps

In [ ]:
if summary:
    for key in ["privacy_sweep", "communication_sweep", "scalability"]:
        rows = summary.get(key, {}).get("rows", [])
        print("\n" + key.upper())
        if not rows:
            print("NOT RUN")
            continue
        frame = pd.DataFrame(rows)
        cols = [c for c in ["privacy_level", "noise_multiplier", "epsilon_estimate", "compression_label", "compression", "clients", "f1", "best_val_f1", "total_bytes", "train_time_s"] if c in frame.columns]
        display(frame[cols])  # noqa: F821


## Figures generated by the suite

In [ ]:
if summary:
    figure_rows = []
    for name, path in summary.get("figures", {}).items():
        p = Path(path) if path else None
        figure_rows.append({"figure": name, "exists": bool(p and p.exists()), "path": str(p) if p else ""})
    display(pd.DataFrame(figure_rows))  # noqa: F821
    for name in ["f1_comparison", "communication_comparison", "training_convergence"]:
        p = summary.get("figures", {}).get(name)
        if p and Path(p).exists():
            display(Image(filename=str(p)))


## Report excerpt

In [ ]:
report = ROOT / "results" / "reports" / "final_experiment_report.md"
if report.exists():
    text = report.read_text(encoding="utf-8")
    display(Markdown(text[:6000]))
else:
    print("Report: NOT RUN. Run the experiment suite first.")


## Claim checklist

- Was the dataset synthetic? Check `dataset.is_synthetic` / source label.
- How many seeds? Check `summary['seeds']`.
- Were sweeps executed? Check stage statuses and rows.
- Is full mode run? It is **NOT RUN** in this repository.

Never copy a number unless it appears in an actual artifact.